[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/milestones/Milestone2_Baseline_and_Representation.ipynb)

> **Run this in Google Colab.** Click the badge above (or open the notebook from Canvas), then run the **Setup** cell first — it installs the dependencies. No local install needed.

# Assignment 2 · Milestone 2 — Project Baseline & Representation

### Team Members

- Zahra Fahimfar
- Krystle Dario
- Varsh Pai

**ISBA 2411 · Due Sunday, July 12, 2026 (11:59 PM PT) · Week 4**

Represent your team's own dataset and establish a first quantitative **baseline** for your problem.

## Deliverables
- **Completed notebook** — this notebook run on *your team's* data (demo data replaced at the swap-in cell).
- **500-word reflection** — in Section 5 of the notebook or a separate PDF.

## What to submit on Canvas
Submit **both** — the live work *and* an archival copy for the record:
1. **GitHub notebook link** — the URL to your team's completed notebook in your project repo (paste it in the *Website URL* box).
2. **PDF copy** — export the completed notebook to PDF (in Colab: *File → Print → Save as PDF*) and upload it (*File Upload*, PDF only).

> Canvas may only accept one submission method per attempt. Either way, **paste your GitHub notebook URL in the cell field below before exporting** so the link is also captured inside the PDF.

**Team GitHub notebook URL:** _(paste here before exporting to PDF)_

**Reading connections**

| Concept | Where to read |
|---|---|
| Tokens & embeddings | HOLLM Ch. 2; J&M Ch. 6 |
| Static vs. contextual representations | HOLLM Ch. 10; Tunstall Ch. 3 |
| Clustering / topic structure | HOLLM Ch. 5 |
| Semantic similarity / search | HOLLM Ch. 8 |

> The techniques below are the **default path**. If your project is classification, extraction, or summarization, adapt the baseline and metric to fit — keep the *represent → baseline → measure* structure.

> **How this notebook works.** It runs **end-to-end on a small built-in demo dataset** so you can see every step working before adapting it. Find the cell marked **`# ====== SWAP IN YOUR TEAM'S DATA HERE ======`** and replace the demo data with your own — everything downstream is written to work off the same variables.

> **Reproducibility.** A fixed random seed is set in the setup cell. On Colab, run the setup cell first (it installs dependencies).

### How this is graded
Scored on the shared milestone rubric ([`docs/ISBA2411_Assignment_Rubric.pdf`](../docs/ISBA2411_Assignment_Rubric.pdf)):

| Rubric criterion | In this milestone |
|---|---|
| **Execution** | Correct TF-IDF + contextual pipelines and a sound baseline on a real task |
| **Analysis & Insight** | Static-vs-contextual comparison + what the baseline number means |
| **Communication** | Clear reflection connecting results to the readings |


In [ ]:
# ===== Setup (Colab) =====
!pip install -q sentence-transformers scikit-learn pandas numpy

import numpy as np, pandas as pd, random
SEED = 42
random.seed(SEED); np.random.seed(SEED)
pd.set_option('display.max_colwidth', 60)

## 0. Data
The demo dataset is a small set of customer-support tickets across three categories (`billing`, `technical`, `shipping`) with a `channel` field used later for a bias check.

**To use your own data:** replace the cell below so that you end up with a DataFrame `df` containing at least a `text` column (and, if you have them, a `label` column for classification and any group column for the bias check).

In [ ]:
# ====== SWAP IN YOUR TEAM'S DATA HERE ======
# Replace this demo block with your own data, e.g.:
#   df = pd.read_csv('data/your_dataset.csv')
#   df = df.rename(columns={'your_text_col': 'text', 'your_label_col': 'label'})
# Keep the variable names below (df / text / label / group) and the rest of the notebook runs.

demo = [
  ('I was charged twice for my subscription this month', 'billing', 'email'),
  ('Why is there an extra fee on my invoice?', 'billing', 'chat'),
  ('My credit card was declined but money was taken', 'billing', 'email'),
  ('Please explain the charges on my latest statement', 'billing', 'email'),
  ('I want a refund for the duplicate payment', 'billing', 'chat'),
  ('The annual plan price is different from what I was quoted', 'billing', 'email'),
  ('How do I update the card on file for billing?', 'billing', 'chat'),
  ('I keep getting billed after cancelling my account', 'billing', 'email'),
  ('The app crashes every time I open the reports page', 'technical', 'chat'),
  ('I cannot log in, it says my password is invalid', 'technical', 'email'),
  ('The export button does nothing when I click it', 'technical', 'chat'),
  ('Getting a 500 error when I try to save settings', 'technical', 'email'),
  ('The mobile app freezes on the loading screen', 'technical', 'chat'),
  ('Two-factor codes never arrive on my phone', 'technical', 'email'),
  ('Charts render blank in the dashboard after update', 'technical', 'chat'),
  ('The API returns malformed JSON for some requests', 'technical', 'email'),
  ('My package has not arrived after two weeks', 'shipping', 'email'),
  ('The tracking number shows no movement for days', 'shipping', 'chat'),
  ('I received the wrong item in my order', 'shipping', 'email'),
  ('Can I change the delivery address before it ships?', 'shipping', 'chat'),
  ('The box arrived damaged and the product is broken', 'shipping', 'email'),
  ('Delivery was marked complete but nothing was delivered', 'shipping', 'chat'),
  ('How long does international shipping usually take?', 'shipping', 'email'),
  ('I need to return an item, what is the shipping process?', 'shipping', 'chat'),
  ('You charged me a late fee I should not owe', 'billing', 'chat'),
  ('The subscription renewed without my consent', 'billing', 'email'),
  ('Dashboard widgets do not load on Safari', 'technical', 'chat'),
  ('Search returns no results even for valid queries', 'technical', 'email'),
  ('Order shipped to my old address by mistake', 'shipping', 'email'),
  ('Where is my refund for the returned package?', 'shipping', 'chat'),
]
df = pd.DataFrame(demo, columns=['text', 'label', 'group'])
texts  = df['text'].tolist()
labels = df['label'].tolist()
print(len(df), 'documents |', df['label'].value_counts().to_dict())
df.head()

## 1. Static representation (baseline)
TF-IDF — fast, interpretable, and the floor your contextual model must beat.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_static = tfidf.fit_transform(texts)
print('TF-IDF matrix:', X_static.shape, '(docs x vocab)')

## 2. Contextual representation
Sentence-Transformers embeddings (`all-MiniLM-L6-v2`, ~90 MB) — dense vectors that capture meaning beyond surface words.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')
X_ctx = embedder.encode(texts, show_progress_bar=False)
print('Contextual embeddings:', X_ctx.shape, '(docs x dim)')

## 3. A task with a metric
We run **both** a task-specific metric and an intrinsic check, on **both** representations:
- **Classification** (task metric): logistic regression, macro-F1 on a held-out split.
- **Clustering** (intrinsic check): KMeans + silhouette score.

Pick whichever is relevant to *your* project; the demo shows both.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, silhouette_score
from sklearn.cluster import KMeans

def eval_classification(X, y, name):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=SEED, stratify=y)
    clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
    pred = clf.predict(Xte)
    return dict(rep=name, accuracy=round(accuracy_score(yte, pred), 3),
                macro_f1=round(f1_score(yte, pred, average='macro'), 3))

def eval_clustering(X, k, name):
    X = X.toarray() if hasattr(X, 'toarray') else X
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(X)
    return dict(rep=name, silhouette=round(silhouette_score(X, km.labels_), 3))

k = df['label'].nunique()
clf_results = pd.DataFrame([eval_classification(X_static, labels, 'TF-IDF (static)'),
                            eval_classification(X_ctx,    labels, 'MiniLM (contextual)')])
clu_results = pd.DataFrame([eval_clustering(X_static, k, 'TF-IDF (static)'),
                            eval_clustering(X_ctx,    k, 'MiniLM (contextual)')])
print('Classification (task metric):'); print(clf_results.to_string(index=False))
print('\nClustering (intrinsic check):'); print(clu_results.to_string(index=False))

## 4. Quick bias check
Probe whether quality differs across a group in your data. The demo compares classification error across the `channel` group (`email` vs `chat`) — large gaps are a fairness red flag.

Adapt this to a group that matters for *your* domain (language, region, demographic proxy, …).

In [ ]:
from sklearn.metrics import accuracy_score

# Train on the contextual rep, then measure accuracy within each group.
idx = np.arange(len(df))
tr, te = train_test_split(idx, test_size=0.4, random_state=SEED, stratify=labels)
clf = LogisticRegression(max_iter=1000).fit(X_ctx[tr], [labels[i] for i in tr])
rows = []
for g in df['group'].unique():
    mask = [i for i in te if df['group'].iloc[i] == g]
    if not mask: continue
    acc = accuracy_score([labels[i] for i in mask], clf.predict(X_ctx[mask]))
    rows.append(dict(group=g, n=len(mask), accuracy=round(acc, 3)))
print(pd.DataFrame(rows).to_string(index=False))
print('\nInterpretation: a large accuracy gap between groups suggests the representation or '
      'model serves one group better — investigate before deploying.')

## 5. Reflection (≈500 words — here or in a separate PDF)
- **Baseline.** What is your baseline number, and on what metric? Why is that the right metric for your problem?
- **Static vs. contextual.** What changed between TF-IDF and embeddings, and *why* (connect to HOLLM Ch. 2/10)?
- **Bias.** What did the group check show, and what would you do about a gap?
- **Next.** What will you try in Milestone 3 to *beat* this baseline?